In [ ]:
import pika
import sys

credentials = pika.PlainCredentials('martin', 'martin00')
parameters =  pika.ConnectionParameters('149.62.71.186', credentials=credentials)
connection = pika.BlockingConnection(parameters)
channel = connection.channel()

channel.exchange_declare(exchange='direct_logs', exchange_type='direct')

result = channel.queue_declare(queue='', exclusive=True)
queue_name = result.method.queue

In [ ]:
channel.queue_bind(exchange='direct_logs', queue=queue_name, routing_key='warning')
channel.queue_bind(exchange='direct_logs', queue=queue_name, routing_key='error')

In [ ]:
def callback(ch, method, properties, body):
    with open("WE.txt", 'a') as file:
        message = body.decode()
        file.write('%r:%r' % (method.routing_key, message))
        file.write('\n')

channel.basic_consume(
    queue=queue_name, on_message_callback=callback, auto_ack=True)

In [ ]:
print('Started consuming')
channel.start_consuming()


#### 1. Connection & Direct Exchange
```python
channel.exchange_declare(exchange='direct_logs', exchange_type='direct')
```
Unlike a simple task queue, this uses a **Direct Exchange**. This acts like a post office that reads a "label" (routing key) on the message and only delivers it to queues that have asked for that specific label.

#### 2. Temporary Anonymous Queue
```python
result = channel.queue_declare(queue='', exclusive=True)
queue_name = result.method.queue
```
* **`queue=''`**: Tells RabbitMQ to generate a random, unique name for the queue.
* **`exclusive=True`**: This makes the queue temporary. As soon as you stop the script, the queue is deleted. This is ideal for logging where you only care about messages while you are actively listening.

#### 3. Binding with Routing Keys (The Filters)
```python
channel.queue_bind(exchange='direct_logs', queue=queue_name, routing_key='warning')
channel.queue_bind(exchange='direct_logs', queue=queue_name, routing_key='error')
```
These lines "subscribe" the queue to specific topics.
* It tells the exchange: "If a message arrives with the label **warning** or **error**, send a copy to my temporary queue."
* If a message with the label `info` is sent, this script will ignore it.



#### 4. The Callback (File Logging)
```python
def callback(ch, method, properties, body):
    with open("WE.txt", 'a') as file:
        message = body.decode()
        file.write('%r:%r' % (method.routing_key, message))
        file.write('\n')
```
When a matching message arrives:
1. It opens `WE.txt` in **append mode** (`'a'`).
2. It writes the **routing key** (so you know if it was a warning or error) followed by the message.
3. It saves the file.

#### 5. Consumption & Auto-Ack
```python
channel.basic_consume(queue=queue_name, on_message_callback=callback, auto_ack=True)
```
* **`auto_ack=True`**: Since logs aren't usually critical "tasks" that need guaranteed processing, the worker tells RabbitMQ the message is received immediately. If the script crashes while writing to the file, that specific message is lost.

---

Would you like me to show you how to send a message to this script using a **Producer** script?